# Feature Engineering — Genre Tags

Builds all tag-based sparse feature matrices from the raw MusicBrainz parquet exports.

**What it does:**
- Builds `album_tags_matrix` — direct album tags, frequency-filtered and per-album normalised
- Builds `artist_tags_matrix` — artist tags, frequency-filtered and per-artist normalised
- Builds `album_genre_matrix` — a blended genre signal using a three-tier label strategy:
  - **Album tags** (w=1.0) — always applied
  - **Artist tags** (w=0.5) — applied to **all albums** via primary artist relationship
  - **Label tags** (w=0.3) — two modes:
    - *Reinforcement:* for albums with existing album/artist signal, label tags are masked to only boost tags already present
    - *New signal:* for albums with zero album and artist signal, label tags from genre-coherent labels (≥60% per-album overlap rate) are allowed to introduce signal

**Why artist tags are universal:** artist tags are specific to the album's own artist and carry valid genre information regardless of how many direct album tags exist. The previous threshold (< 5 direct tags) was unnecessarily conservative.

**Why the label allowlist exists:** large labels with diverse catalogues produce tag noise when applied unconditionally. The allowlist identifies labels where ≥60% of their albums share at least one tag with the label — a proxy for genre coherence. Only these labels are trusted to introduce new signal for zero-coverage albums.

**Inputs:** `data/features/album_ids.pkl`, `data/features/artist_ids.pkl`,
`data/mb_album_tag.parquet`, `data/mb_artist_tag.parquet`, `data/mb_album_label.parquet`,
`data/mb_album_artists.parquet`

**Outputs to `data/features/`:** `album_tags_matrix.npz`, `artist_tags_matrix.npz`, `album_genre_matrix.npz`

**Run after:** `01-album-artist-index.ipynb`

## Imports & Constants

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.preprocessing import normalize

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

MIN_TAG_OCC          = 10   # tags appearing fewer times than this across all sources are dropped
W_ALBUM              = 1.0
W_ARTIST             = 0.5
W_LABEL              = 0.3
LABEL_OVERLAP_THRESHOLD = 0.6  # min fraction of a label's albums that must share a tag with the label

## Load Master ID Index

Row order for every matrix built in this notebook is determined by `album_ids.pkl` and `artist_ids.pkl`, saved by `01-album-artist-index.ipynb`. Loading them here (rather than re-deriving from parquet) guarantees this notebook produces matrices that are row-aligned with every other feature matrix.

In [ ]:
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)
album_index = pd.Index(album_ids)
n_albums = len(album_index)

with open(f'{FEATURES_DIR}/artist_ids.pkl', 'rb') as f:
    artist_ids = pickle.load(f)
artist_index = pd.Index(artist_ids)

print(f'Album universe : {n_albums:,}')
print(f'Artist universe: {len(artist_index):,}')

## Load Tag Sources

In [ ]:
album_tags  = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
artist_tags = pd.read_parquet(f'{DATA_DIR}/mb_artist_tag.parquet')
album_label = pd.read_parquet(f'{DATA_DIR}/mb_album_label.parquet')

# Keep only positive-vote tags
album_tags  = album_tags[album_tags['tag_count'] > 0].copy()
artist_tags = artist_tags[artist_tags['tag_count'] > 0].copy()
album_label = album_label[album_label['tag_count'] > 0].copy()

print(f'Album tag rows : {len(album_tags):,}')
print(f'Artist tag rows: {len(artist_tags):,}')
print(f'Label tag rows : {len(album_label):,}')

## Build Artist Tags Sparse Matrix

Constructs the artist-by-tag feature matrix:

1. **Tag frequency filter (`>= 10`):** Tags appearing on fewer than 10 artists are dropped — they carry little generalisation signal and inflate column count.
2. **Per-artist weight normalisation:** Each artist's raw tag counts are divided by their total, producing relative weights in [0, 1].
3. **COO coordinate generation:** `artist_index.get_indexer` maps artist IDs to row positions in the full universe; artists absent from the index return -1 and are excluded.
4. **CSR construction:** Explicit `shape` anchors the matrix to the full artist universe so artists with no tags have all-zero rows rather than being absent.

In [ ]:
print('1. Filtering rare artist tags...')
artist_tag_counts = artist_tags.groupby('tag_id').size()
popular_artist_tags = artist_tag_counts[artist_tag_counts >= MIN_TAG_OCC].index
artist_tags_filtered = artist_tags[artist_tags['tag_id'].isin(popular_artist_tags)].copy()

print('2. Normalising artist tag weights...')
artist_totals = artist_tags_filtered.groupby('artist_id')['tag_count'].transform('sum')
artist_tags_filtered['tag_weight'] = (artist_tags_filtered['tag_count'] / artist_totals).astype('float32')

print('3. Generating category codes...')
artist_tags_filtered['tag_code'] = artist_tags_filtered['tag_id'].astype('category').cat.codes
unique_artist_tag_ids = artist_tags_filtered['tag_id'].astype('category').cat.categories
artist_tags_filtered['artist_code'] = artist_index.get_indexer(artist_tags_filtered['artist_id'])

print('4. Building sparse matrix...')
valid = artist_tags_filtered['artist_code'] >= 0
X_artist_tags = csr_matrix(
    (artist_tags_filtered.loc[valid, 'tag_weight'].values,
     (artist_tags_filtered.loc[valid, 'artist_code'].values,
      artist_tags_filtered.loc[valid, 'tag_code'].values)),
    shape=(len(artist_index), len(unique_artist_tag_ids))
)

print(f'Artist tags matrix: {X_artist_tags.shape}  nnz={X_artist_tags.nnz:,}')

## Build Album Tags Sparse Matrix

Identical pipeline to the artist tags cell above, applied to albums. `unique_tag_ids` and `unique_artist_tag_ids` are independent vocabularies — album and artist tagging activity in MusicBrainz are separate and produce different column spaces.

In [ ]:
print('1. Filtering rare album tags...')
tag_counts = album_tags.groupby('tag_id').size()
popular_tags = tag_counts[tag_counts >= MIN_TAG_OCC].index
album_tags_filtered = album_tags[album_tags['tag_id'].isin(popular_tags)].copy()

print('2. Normalising tag weights...')
album_totals = album_tags_filtered.groupby('album_id')['tag_count'].transform('sum')
album_tags_filtered['tag_weight'] = (album_tags_filtered['tag_count'] / album_totals).astype('float32')

print('3. Generating category codes...')
album_tags_filtered['tag_code'] = album_tags_filtered['tag_id'].astype('category').cat.codes
unique_tag_ids = album_tags_filtered['tag_id'].astype('category').cat.categories
album_tags_filtered['album_code'] = album_index.get_indexer(album_tags_filtered['album_id'])

print('4. Building sparse matrix...')
valid = album_tags_filtered['album_code'] >= 0
X_album_tags = csr_matrix(
    (album_tags_filtered.loc[valid, 'tag_weight'].values,
     (album_tags_filtered.loc[valid, 'album_code'].values,
      album_tags_filtered.loc[valid, 'tag_code'].values)),
    shape=(n_albums, len(unique_tag_ids))
)

print(f'Album tags matrix: {X_album_tags.shape}  nnz={X_album_tags.nnz:,}')

## Build Genre Tag Vocabulary

The genre matrix uses a unified vocabulary built across all three tag sources (album, artist, label) combined. Only tags with >= `MIN_TAG_OCC` total occurrences across sources are kept. Building the vocabulary once here ensures the column space is consistent across all three blocks.

In [ ]:
album_artists = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id'])
    .drop_duplicates(subset='album_id')
)

artist_tags_on_albums = (
    album_artists
    .merge(artist_tags, on='artist_id', how='inner')
    [['album_id', 'tag_id', 'tag_count']]
)

# Keep label_id so we can filter to the allowlist later
label_tags = album_label[['album_id', 'label_id', 'tag_id', 'tag_count']].copy()

all_tags = pd.concat([
    album_tags[['tag_id', 'tag_count']],
    artist_tags_on_albums[['tag_id', 'tag_count']],
    label_tags[['tag_id', 'tag_count']],
], ignore_index=True)

tag_occ   = all_tags.groupby('tag_id')['tag_count'].sum()
genre_tag_index = pd.Index(sorted(tag_occ[tag_occ >= MIN_TAG_OCC].index))
n_tags = len(genre_tag_index)

print(f'Total unique tags across all sources: {tag_occ.shape[0]:,}')
print(f'Tags kept (>= {MIN_TAG_OCC} occurrences)        : {n_tags:,}')

## Build Label Genre Coherence Allowlist

Identifies labels whose tags are genuinely predictive of the genre of their individual albums.
For each label that has any tags, we compute the fraction of its albums where at least one
of the label's tags also appears on that album directly. Labels meeting `LABEL_OVERLAP_THRESHOLD`
(60%) are added to the allowlist.

This computation uses a merge-based approach: join label-to-album relationships onto label tag
rows, then inner-join onto album tags to find (label, album) pairs with at least one matching tag.
The overlap rate is `matching_albums / total_albums_on_label_with_any_album_tags`.

In [ ]:
# Labels that carry any tag data
label_tag_rows = album_label[['label_id', 'tag_id']].drop_duplicates()
labels_with_tags = set(label_tag_rows['label_id'].unique())

# (album, label) deduplicated pairs — one row per relationship
al_deduped = album_label.drop_duplicates(subset=['album_id', 'label_id'])[['label_id', 'album_id']]
al_relevant = al_deduped[al_deduped['label_id'].isin(labels_with_tags)]

# Denominator: albums on the label that also have direct album tags
albums_with_direct_tags = set(album_tags['album_id'].unique())
al_comparable = al_relevant[al_relevant['album_id'].isin(albums_with_direct_tags)]
total_per_label = al_comparable.groupby('label_id')['album_id'].nunique()

# Albums where the label's tag appears in that album's own tags
overlap_check = (
    al_comparable
    .merge(label_tag_rows, on='label_id', how='inner')       # expand to (label, album, tag) triples
    .merge(                                                    # inner join: keep only matching album tags
        album_tags[['album_id', 'tag_id']].drop_duplicates(),
        on=['album_id', 'tag_id'], how='inner'
    )
    [['label_id', 'album_id']]
    .drop_duplicates()
)
overlapping_per_label = overlap_check.groupby('label_id')['album_id'].nunique()

per_label_overlap = (
    pd.DataFrame({'total': total_per_label, 'overlapping': overlapping_per_label})
    .fillna(0)
    .assign(overlap_rate=lambda d: d['overlapping'] / d['total'])
)

label_allowlist = set(per_label_overlap[per_label_overlap['overlap_rate'] >= LABEL_OVERLAP_THRESHOLD].index)

print(f'Labels assessed : {len(per_label_overlap):,}')
print(f'Allowlist (>={LABEL_OVERLAP_THRESHOLD*100:.0f}% overlap): {len(label_allowlist):,}')
print(f'Excluded        : {len(per_label_overlap) - len(label_allowlist):,}')

## Build Genre Matrix — Three Blocks

Album and artist blocks are built first and summed into `X_genre_base`. Label tags are then
applied in two modes depending on whether the album already has genre signal:

- **Reinforcement** (all albums): label tags masked to `X_genre_base > 0` — only boosts tags
  the album or artist already has. Prevents diverse labels from injecting unrelated genre tags.
- **New signal** (zero-signal albums only): label tags from allowlist labels only — these labels
  have demonstrated ≥60% per-album genre coherence so their tags are trusted to introduce signal
  for albums that have none from any other source.

L1 normalisation after combining brings each album row to sum 1.0.

In [ ]:
# Block 1: Album tags
at = album_tags[album_tags['tag_id'].isin(genre_tag_index)].copy()
at_totals = at.groupby('album_id')['tag_count'].transform('sum')
at['weight'] = (at['tag_count'] / at_totals * W_ALBUM).astype('float32')

row_idx = album_index.get_indexer(at['album_id'].values)
col_idx = genre_tag_index.get_indexer(at['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)

X_genre_album = csr_matrix(
    (at['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Album block : {X_genre_album.shape}  nnz={X_genre_album.nnz:,}')

In [ ]:
# Block 2: Artist tags — all albums via primary artist relationship
art = (
    album_artists
    .merge(artist_tags[artist_tags['tag_id'].isin(genre_tag_index)], on='artist_id', how='inner')
    [['album_id', 'tag_id', 'tag_count']]
)

art_totals = art.groupby('album_id')['tag_count'].transform('sum')
art['weight'] = (art['tag_count'] / art_totals * W_ARTIST).astype('float32')

row_idx = album_index.get_indexer(art['album_id'].values)
col_idx = genre_tag_index.get_indexer(art['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)

X_genre_artist = csr_matrix(
    (art['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Artist block: {X_genre_artist.shape}  nnz={X_genre_artist.nnz:,}')

In [ ]:
# Block 3: Label tags — build two variants
# 3a: all labels (used for reinforcement masking against existing signal)
# 3b: allowlist labels only (used to introduce new signal for zero-signal albums)

lt = label_tags[label_tags['tag_id'].isin(genre_tag_index)].copy()
lt_totals = lt.groupby('album_id')['tag_count'].transform('sum')
lt['weight'] = (lt['tag_count'] / lt_totals * W_LABEL).astype('float32')

row_idx = album_index.get_indexer(lt['album_id'].values)
col_idx = genre_tag_index.get_indexer(lt['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)

# 3a: all labels
X_genre_label_raw = csr_matrix(
    (lt['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Label block (all)      : {X_genre_label_raw.shape}  nnz={X_genre_label_raw.nnz:,}')

# 3b: allowlist labels only — re-normalise weights within allowlist labels per album
lt_allow = lt[lt['label_id'].isin(label_allowlist)].copy()
lt_allow_totals = lt_allow.groupby('album_id')['tag_count'].transform('sum')
lt_allow['weight'] = (lt_allow['tag_count'] / lt_allow_totals * W_LABEL).astype('float32')

row_idx_a = album_index.get_indexer(lt_allow['album_id'].values)
col_idx_a = genre_tag_index.get_indexer(lt_allow['tag_id'].values)
valid_a   = (row_idx_a >= 0) & (col_idx_a >= 0)

X_genre_label_allowlist = csr_matrix(
    (lt_allow['weight'].values[valid_a], (row_idx_a[valid_a], col_idx_a[valid_a])),
    shape=(n_albums, n_tags)
)
print(f'Label block (allowlist): {X_genre_label_allowlist.shape}  nnz={X_genre_label_allowlist.nnz:,}')

In [ ]:
# Tier 1 + 2: album tags + artist tags (universal)
X_genre_base = X_genre_album + X_genre_artist

# Tier 3a: reinforcement — label tags masked to existing signal only
signal_mask = X_genre_base > 0
X_label_reinforcement = X_genre_label_raw.multiply(signal_mask)

# Tier 3b: new signal for zero-signal albums — allowlist labels only
# Multiply by a column vector: 1.0 for rows with no existing signal, 0.0 otherwise
no_signal_rows = (np.diff(X_genre_base.indptr) == 0).astype('float32').reshape(-1, 1)
X_label_zero_signal = X_genre_label_allowlist.multiply(no_signal_rows)

# Combine all tiers and L1-normalise
X_genre = X_genre_base + X_label_reinforcement + X_label_zero_signal
X_genre = normalize(X_genre, norm='l1', axis=1)

reinforcement_kept = X_label_reinforcement.nnz
reinforcement_dropped = X_genre_label_raw.nnz - reinforcement_kept
zero_signal_added = X_label_zero_signal.nnz

print(f'Label reinforcement entries kept   : {reinforcement_kept:,}')
print(f'Label entries masked as noise      : {reinforcement_dropped:,}  ({reinforcement_dropped / X_genre_label_raw.nnz * 100:.1f}%)')
print(f'Label entries added (zero-signal)  : {zero_signal_added:,}')
print(f'Genre matrix : {X_genre.shape}  nnz={X_genre.nnz:,}')
print(f'Albums with any genre signal: {(np.diff(X_genre.indptr) > 0).sum():,}')

## Save Matrices

In [ ]:
save_npz(f'{FEATURES_DIR}/album_tags_matrix.npz',  X_album_tags)
save_npz(f'{FEATURES_DIR}/album_genre_matrix.npz', X_genre)

print(f'album_tags_matrix  : {X_album_tags.shape}')
print(f'album_genre_matrix : {X_genre.shape}')
print('Done.')